# Lecture 14: Self-Consistency and Verified Reasoning with the Claude API

### Turning a single model call into a more reliable reasoning system

Lectures 12–13 stayed in *embedding space* — measuring meaning with inner
products. This lecture closes the loop with a **live language model (Claude)**
and applies the reasoning techniques from `METHODOLOGY.md` (Tier A #3–#4):

1. **Self-consistency** — sample several independent reasoning paths and
   aggregate them, instead of trusting one sample. Geometrically: find the
   densest region of answer-space (Lecture 13's clustering idea).
2. **Verification** — a second, independent model pass that critically checks
   the candidate answer before you accept it.
3. **Decomposition** — break a problem into typed sub-steps that compose
   cleanly, the way matrices compose (Lecture 8).

> **Runs live against the Claude API.** Every cell that calls the model is
> guarded by an API-key check, so the notebook executes top-to-bottom even
> without a key (it prints a notice). Set `ANTHROPIC_API_KEY` to get real
> results.

## Setup

We use the official `anthropic` SDK and the current flagship model,
`claude-opus-4-8`, with **adaptive thinking** (the model decides how much to
reason per request).

> **An important real constraint:** on Opus 4.8 the `temperature` / `top_p` /
> `top_k` sampling parameters are **removed** (they return a 400 error). The
> classic way to get *diverse* self-consistency samples — raising the
> temperature — is therefore unavailable. Instead we induce diversity the
> robust way: by varying the **framing** of each sample's prompt (solve
> algebraically vs. by estimation vs. working backwards). This is a more
> honest source of reasoning diversity anyway.

In [1]:
import os, re, collections

MODEL = 'claude-opus-4-8'
HAS_KEY = bool(os.environ.get('ANTHROPIC_API_KEY'))

if HAS_KEY:
    import anthropic
    client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment
    print('Anthropic client ready. Model:', MODEL)
else:
    client = None
    print('No ANTHROPIC_API_KEY set — live cells will print a notice and skip.')
    print('To run for real:  export ANTHROPIC_API_KEY=sk-ant-...  then re-run.')

No ANTHROPIC_API_KEY set — live cells will print a notice and skip.
To run for real:  export ANTHROPIC_API_KEY=sk-ant-...  then re-run.


## A reasoning problem

We use a small word problem with a single correct answer so we can *measure*
whether the techniques help. (The correct answer is 29.)

In [2]:
PROBLEM = (
    'A library has 5 shelves. Each shelf holds 8 books. '
    'On Monday 12 books are borrowed and 3 are returned. '
    'On Tuesday 7 more are borrowed. '
    'How many books are on the shelves at the end of Tuesday?'
)
GROUND_TRUTH = '29'   # 40 - 12 + 3 - 7
print(PROBLEM)

A library has 5 shelves. Each shelf holds 8 books. On Monday 12 books are borrowed and 3 are returned. On Tuesday 7 more are borrowed. How many books are on the shelves at the end of Tuesday?


## Structured output: extract a clean answer

We ask the model to return a **structured** result (a final answer plus a
one-line justification) using the SDK's schema-validated parsing, so we never
have to scrape free text. We define the schema with Pydantic.

In [3]:
from pydantic import BaseModel

class Solution(BaseModel):
    final_answer: str        # just the number
    one_line_reason: str     # a brief justification

# Different 'framings' create diverse reasoning paths without a temperature knob.
FRAMINGS = [
    'Solve this step by step, tracking the running total after each event.',
    'Solve this by first computing the starting total, then applying each change.',
    'Solve this by working out the net change across both days, then adding it to the start.',
    'Estimate the answer first, then verify it precisely.',
    'Solve it carefully and double-check your arithmetic before answering.',
]

def one_path(framing):
    """Generate one reasoning path; return a validated Solution."""
    resp = client.messages.parse(
        model=MODEL,
        max_tokens=2000,
        thinking={'type': 'adaptive'},
        system='You are a careful problem solver. Return only the requested fields.',
        messages=[{'role': 'user', 'content': f'{framing}\n\nProblem: {PROBLEM}'}],
        output_format=Solution,
    )
    return resp.parsed_output

## 1. Self-consistency: sample, then take the majority

We run several framings, normalize each final answer, and take the **majority
vote**. A single sample can slip on arithmetic; the consensus across
independent paths is far more reliable.

(For *free-text* answers rather than a number, you would embed the answers and
cluster them — exactly Lecture 13 — and take the centroid of the largest
cluster. Here a number lets us use exact-match voting, which is cleaner.)

In [4]:
def normalize(ans):
    m = re.search(r'-?\d+', ans.replace(',', ''))
    return m.group(0) if m else ans.strip().lower()

def self_consistent_answer():
    solutions = [one_path(f) for f in FRAMINGS]
    votes = collections.Counter(normalize(s.final_answer) for s in solutions)
    winner, count = votes.most_common(1)[0]
    return winner, count, len(solutions), solutions

if HAS_KEY:
    winner, count, n, sols = self_consistent_answer()
    for i, s in enumerate(sols):
        print(f'path {i+1}: {normalize(s.final_answer):>4}   ({s.one_line_reason})')
    print(f'\nmajority answer: {winner}  ({count}/{n} paths agreed)')
    print('correct?' , winner == GROUND_TRUTH)
else:
    print('[skipped — set ANTHROPIC_API_KEY] would sample',
          len(FRAMINGS), 'reasoning paths and majority-vote the final answer.')

[skipped — set ANTHROPIC_API_KEY] would sample 5 reasoning paths and majority-vote the final answer.


## 2. Verification: an independent check before accepting

Self-consistency tells us what the model *agrees on*; it doesn't tell us if the
agreement is *correct*. A **verifier** is a second, independent call whose only
job is to check the candidate answer skeptically and report a verdict.

In [5]:
class Verdict(BaseModel):
    is_correct: bool
    explanation: str

def verify(candidate):
    resp = client.messages.parse(
        model=MODEL,
        max_tokens=2000,
        thinking={'type': 'adaptive'},
        system='You are a skeptical checker. Recompute the answer independently '
               'and decide whether the candidate is correct.',
        messages=[{'role': 'user',
                   'content': f'Problem: {PROBLEM}\n\nCandidate answer: {candidate}\n'
                              'Is this correct? Recompute from scratch.'}],
        output_format=Verdict,
    )
    return resp.parsed_output

if HAS_KEY:
    v = verify(winner)
    print('candidate:', winner)
    print('verifier says correct?', v.is_correct)
    print('explanation:', v.explanation)
    accepted = winner if v.is_correct else None
    print('\nACCEPTED answer:', accepted)
else:
    print('[skipped — set ANTHROPIC_API_KEY] would independently recompute and',
          'either confirm or reject the majority answer.')

[skipped — set ANTHROPIC_API_KEY] would independently recompute and either confirm or reject the majority answer.


## 3. Decomposition: typed sub-steps that compose

Instead of one big leap, we can have the model solve a problem as a chain of
small, checkable steps — each step's output feeding the next, the way
composing matrices $C = BA$ is cleaner than one tangled operator (Lecture 8).
This is most useful for problems too complex for a single reliable pass.

In [6]:
def ask(prompt, system='Answer with just the requested value.'):
    resp = client.messages.create(
        model=MODEL, max_tokens=1500, thinking={'type': 'adaptive'},
        system=system,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return ''.join(b.text for b in resp.content if b.type == 'text').strip()

if HAS_KEY:
    start = ask(f'{PROBLEM}\nStep 1 only: how many books are on the shelves to begin with? Number only.')
    monday = ask(f'{PROBLEM}\nStarting total is {start}. Step 2 only: the total after Mondays borrows and returns? Number only.')
    tuesday = ask(f'{PROBLEM}\nAfter Monday the total is {monday}. Step 3 only: the total after Tuesday? Number only.')
    print(f'start={normalize(start)}  ->  after Mon={normalize(monday)}  ->  after Tue={normalize(tuesday)}')
    print('correct?', normalize(tuesday) == GROUND_TRUTH)
else:
    print('[skipped — set ANTHROPIC_API_KEY] would chain three typed sub-steps,',
          'each feeding the next, to reach the final total.')

[skipped — set ANTHROPIC_API_KEY] would chain three typed sub-steps, each feeding the next, to reach the final total.


## Recap

Three reasoning techniques, each a thin wrapper around ordinary model calls:

| Technique | What it buys you | The idea from earlier lectures |
|-----------|------------------|--------------------------------|
| Self-consistency | Robustness to one-off slips | Densest cluster in answer-space (Lec. 13) |
| Verification | Catches confident-but-wrong answers | An independent second opinion |
| Decomposition | Reliability on complex problems | Clean composition (Lec. 8) |

Combined, they turn a single fallible call into a small reasoning *system*:
**sample several paths → take the consensus → verify it → (if needed) decompose**.
None of this requires access to the model's internals — it's all orchestration
around the API, which is exactly what makes it usable today (Tier A of the
methodology).

### A note on Opus 4.8 specifics (so your code actually runs)
- Model id: `claude-opus-4-8`; thinking via `{'type': 'adaptive'}`.
- **No `temperature`/`top_p`/`top_k`** — they 400. Get sample diversity from
  prompt framing (as above), not a temperature knob.
- Use **structured outputs** (`messages.parse` + a Pydantic schema) to extract
  clean answers instead of scraping text.

## Exercises

1. Increase the number of framings to 9 and plot how often the majority answer
   is correct over several runs. Does more sampling help?
2. Make the verifier return a numeric confidence and only accept answers above a
   threshold; route low-confidence cases to decomposition.
3. Swap the numeric problem for a free-text question and replace majority voting
   with **embedding-based clustering** from Lecture 13 (embed the answers, take
   the centroid of the largest cluster).
4. Add a second, differently-framed verifier and accept only when both agree
   (an adversarial 2-of-2 check).
5. Measure cost: print `resp.usage` and compare the token cost of
   self-consistency-plus-verification against a single call. When is the extra
   reliability worth it?